In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from openai import OpenAI
from rag_helper_hw import RAGBase

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

Q2. Indexing and searching

In [5]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

In [6]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(question, num_results=1)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

Q3. RAG

In [7]:
openai_client = OpenAI()

In [9]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer, usage = assistant.rag(question)
print(answer)
print(f"Input Tokens: {usage.input_tokens}")

It keeps calling the model inside a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there is a function call, the code runs the tool, appends the tool output to the message history, and loops again.
- If there are no function calls in that turn, it breaks out of the loop.

So the stop condition is: **no function calls in the model’s response**.
Input Tokens: 7141


Q4. Chunking

In [10]:
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

Q5. RAG with chunking

In [11]:
chunks_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunks_index.fit(chunks)

In [12]:
assistant = RAGBase(
    index=chunks_index,
    llm_client=openai_client,
)

answer, usage = assistant.rag(question)
print(answer)
print(f"Input Tokens: {usage.input_tokens}")

It keeps doing this with a `while True` loop:

- Call the model.
- Check the response for any `function_call` items.
- If there are tool calls, run them, append the tool outputs to `messages`, and loop again.
- If there are no function calls in that turn, `break`.

So the stop condition is: **no function calls returned in the current model response**.
Input Tokens: 2324


Q6. Turning it into an agent

In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
def search(query: str) -> dict[str, str]:
    """
    Search the course content database for entries matching the given query.
    """
    return index.search(query, num_results=5)


instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

question = "How does the agentic loop work, and how is it different from plain RAG?"

In [15]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [16]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [17]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

In [18]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received
